# Seq2Seq LSTM (Encoder–Decoder) using Keras
This notebook is a complete implementation based on the official Keras example.
Dataset: ManyThings English–French (`fra.txt`).

In [ ]:
# Install (if needed)
# !pip install tensorflow keras matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Input,LSTM,Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import get_file
import os

In [ ]:
# Hyperparameters
batch_size=64
epochs=20
latent_dim=256
num_samples=10000

In [ ]:
# Download dataset
path=get_file("fra-eng.zip",
              origin="https://www.manythings.org/anki/fra-eng.zip",
              extract=True)

data_path=os.path.join(os.path.dirname(path),"fra-eng","fra.txt")
print(data_path)

In [ ]:
# Read dataset
input_texts=[]
target_texts=[]

input_chars=set()
target_chars=set()

with open(data_path,encoding="utf-8") as f:
    lines=f.read().split("\n")

for line in lines[:min(num_samples,len(lines)-1)]:
    parts=line.split("\t")
    if len(parts)<2:
        continue
    inp=parts[0]
    tar="\t"+parts[1]+"\n"
    input_texts.append(inp)
    target_texts.append(tar)
    input_chars.update(inp)
    target_chars.update(tar)

input_chars=sorted(list(input_chars))
target_chars=sorted(list(target_chars))

num_encoder_tokens=len(input_chars)
num_decoder_tokens=len(target_chars)

max_encoder_seq_length=max(len(x) for x in input_texts)
max_decoder_seq_length=max(len(x) for x in target_texts)

input_token_index={c:i for i,c in enumerate(input_chars)}
target_token_index={c:i for i,c in enumerate(target_chars)}

print("Samples:",len(input_texts))

In [ ]:
# One-hot encoding
encoder_input_data=np.zeros((len(input_texts),max_encoder_seq_length,num_encoder_tokens),dtype="float32")
decoder_input_data=np.zeros((len(input_texts),max_decoder_seq_length,num_decoder_tokens),dtype="float32")
decoder_target_data=np.zeros((len(input_texts),max_decoder_seq_length,num_decoder_tokens),dtype="float32")

for i,(inp,tar) in enumerate(zip(input_texts,target_texts)):
    for t,ch in enumerate(inp):
        encoder_input_data[i,t,input_token_index[ch]]=1.
    for t,ch in enumerate(tar):
        decoder_input_data[i,t,target_token_index[ch]]=1.
        if t>0:
            decoder_target_data[i,t-1,target_token_index[ch]]=1.


In [ ]:
# Build Seq2Seq model
encoder_inputs=Input(shape=(None,num_encoder_tokens))
encoder=LSTM(latent_dim,return_state=True)
_,state_h,state_c=encoder(encoder_inputs)
encoder_states=[state_h,state_c]

decoder_inputs=Input(shape=(None,num_decoder_tokens))
decoder=LSTM(latent_dim,return_sequences=True,return_state=True)
decoder_outputs,_,_=decoder(decoder_inputs,initial_state=encoder_states)

dense=Dense(num_decoder_tokens,activation='softmax')
decoder_outputs=dense(decoder_outputs)

model=Model([encoder_inputs,decoder_inputs],decoder_outputs)
model.compile(optimizer='rmsprop',
              loss='categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

In [ ]:
history=model.fit(
    [encoder_input_data,decoder_input_data],
    decoder_target_data,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.2
)

In [ ]:
plt.plot(history.history['loss'],label='Train')
plt.plot(history.history['val_loss'],label='Validation')
plt.legend()
plt.show()

In [ ]:
# Inference models
encoder_model=Model(encoder_inputs,encoder_states)

decoder_state_input_h=Input(shape=(latent_dim,))
decoder_state_input_c=Input(shape=(latent_dim,))
decoder_states_inputs=[decoder_state_input_h,decoder_state_input_c]

decoder_outputs,state_h,state_c=decoder(
    decoder_inputs,
    initial_state=decoder_states_inputs
)

decoder_states=[state_h,state_c]
decoder_outputs=dense(decoder_outputs)

decoder_model=Model(
    [decoder_inputs]+decoder_states_inputs,
    [decoder_outputs]+decoder_states
)


In [ ]:
reverse_target_char_index={i:c for c,i in target_token_index.items()}

def decode_sequence(input_seq):

    states=encoder_model.predict(input_seq,verbose=0)

    target_seq=np.zeros((1,1,num_decoder_tokens))
    target_seq[0,0,target_token_index["\t"]]=1.

    decoded=""

    while True:

        output,h,c=decoder_model.predict([target_seq]+states,verbose=0)

        idx=np.argmax(output[0,-1,:])
        char=reverse_target_char_index[idx]
        decoded+=char

        if char=="\n" or len(decoded)>max_decoder_seq_length:
            break

        target_seq=np.zeros((1,1,num_decoder_tokens))
        target_seq[0,0,idx]=1.
        states=[h,c]

    return decoded.strip()


In [ ]:
# Test
for i in range(10):
    pred=decode_sequence(encoder_input_data[i:i+1])
    print("Input :",input_texts[i])
    print("Target:",target_texts[i].strip())
    print("Pred  :",pred)
    print("-"*50)


## Custom Prediction
Encode a custom English sentence similarly to the preprocessing stage, then call `decode_sequence()`.